# Customer Segmentation Project 
## Machine Learning II 

A K M SAIF HOQUE(20241418)


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.cluster import KMeans,DBSCAN,AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_style('whitegrid')

In [5]:
customer_info=pd.read_csv("../data/customer_info.csv")
customer_basket=pd.read_csv("../data/customer_basket.csv")

In [7]:
customer_info.shape

(33038, 25)

In [8]:
customer_basket.shape

(100000, 3)

In [9]:
customer_info.head()

,customer_id,customer_name,customer_gender,customer_birthdate,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,...,lifetime_spend_fish,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude
0,3,Bsc. Crystal Kitchens,female,02/12/1970 01:36 PM,1.0,1.0,1.0,3.0,11731.0,4553.0,...,213.0,552.0,256.0,384.0,189.0,0.631599,2020.0,1.0,38.794428,-9.215739
1,4,Bsc. Glenda Bauman,female,11/13/1975 06:58 PM,1.0,0.0,0.0,2.0,13694.0,963.0,...,15.0,1880.0,333.0,665.0,130.0,0.149890,2013.0,1.0,38.751711,-9.179611
2,5,Msc. Antonio Campbell,male,09/10/1971 10:07 AM,0.0,0.0,NaN,2.0,12407.0,0.0,...,273.0,507.0,101.0,222.0,81.0,0.069126,2005.0,NaN,38.780678,-9.160656
3,7,John Kelling,male,10/23/1982 11:20 AM,0.0,0.0,2.0,1.0,7493.0,1105.0,...,1083.0,485.0,1656.0,184.0,92.0,0.253609,2021.0,1.0,38.739548,-9.148679
4,8,Arthur Dematteo,male,08/04/1969 10:22 PM,0.0,0.0,3.0,1.0,9187.0,10841.0,...,1015.0,297.0,1258.0,441.0,6.0,0.186569,2021.0,1.0,38.733071,-9.188188


In [10]:
customer_basket.head()

,invoice_id,list_of_goods,customer_id
0,3700630,"['chicken', 'rice', 'pepper', 'whole wheat ric...",12912
1,10242376,"['low fat yogurt', 'tomatoes', 'pepper', 'aspa...",22853
2,91550,"['cake', 'tomatoes', 'pancakes', 'iPad', 'fina...",19
3,3137503,"['cereals', 'megaman zero', 'final fantasy XIX...",10995
4,7165061,"['rice', 'frozen smoothie', 'black tea', 'tea'...",27807


In [11]:
customer_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33038 entries, 0 to 33037
Data columns (total 25 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   customer_id                              33038 non-null  int64  
 1   customer_name                            33038 non-null  object 
 2   customer_gender                          33038 non-null  object 
 3   customer_birthdate                       32873 non-null  object 
 4   kids_home                                32708 non-null  float64
 5   teens_home                               32708 non-null  float64
 6   number_complaints                        32377 non-null  float64
 7   distinct_stores_visited                  32708 non-null  float64
 8   lifetime_spend_groceries                 33038 non-null  float64
 9   lifetime_spend_electronics               32377 non-null  float64
 10  typical_hour                             32377

In [12]:
customer_info.describe()

,customer_id,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,typical_hour,lifetime_spend_vegetables,lifetime_spend_nonalcohol_drinks,...,lifetime_spend_fish,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude
count,33038.000000,32708.000000,32708.000000,32377.000000,32708.000000,33038.000000,32377.000000,32377.000000,32377.000000,33038.000000,...,32047.000000,32708.00000,32377.000000,32377.000000,33038.000000,32708.000000,33038.000000,19932.0,33038.000000,33038.000000
mean,19974.265785,1.116118,0.898893,0.930846,3.167941,16306.227798,2763.080088,12.659388,727.223801,464.352776,...,608.781228,820.34646,373.900917,336.217099,148.914644,0.318866,2015.311853,1.0,38.749694,-9.154549
std,11538.538632,1.150186,0.962924,0.894658,1.674114,11985.903518,3453.191495,4.854708,654.633087,275.767976,...,497.068874,608.31732,460.782042,160.234980,105.922907,0.283638,5.032196,0.0,0.022498,0.028581
min,3.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,6.000000,0.000000,0.000000,...,0.000000,0.00000,0.000000,0.000000,0.000000,-1.274940,1993.000000,1.0,38.687987,-9.232989
25%,9985.250000,0.000000,0.000000,0.000000,2.000000,8647.000000,579.000000,8.000000,224.000000,241.000000,...,171.000000,362.00000,125.000000,225.000000,67.000000,0.122359,2012.000000,1.0,38.734079,-9.173732
50%,19951.500000,1.000000,1.000000,1.000000,3.000000,13002.500000,1470.000000,12.000000,471.000000,421.000000,...,511.000000,686.00000,223.000000,327.000000,123.000000,0.239449,2015.000000,1.0,38.748286,-9.156689
75%,29964.750000,1.000000,1.000000,1.000000,4.000000,20807.000000,3745.000000,16.000000,1074.000000,640.000000,...,923.000000,1120.00000,374.000000,435.000000,210.000000,0.469390,2019.000000,1.0,38.765779,-9.139608
max,40000.000000,8.000000,6.000000,7.000000,10.000000,104670.000000,35299.000000,23.000000,3337.000000,2180.000000,...,3172.000000,3482.00000,3936.000000,1224.000000,600.000000,1.000000,2029.000000,1.0,38.823693,-9.035697


In [13]:
customer_info.isnull().sum().sort_values(ascending=False)

loyalty_card_number                        13106
lifetime_spend_fish                          991
lifetime_spend_videogames                    661
typical_hour                                 661
lifetime_spend_meat                          661
lifetime_spend_electronics                   661
lifetime_spend_vegetables                    661
lifetime_spend_petfood                       661
number_complaints                            661
lifetime_spend_alcohol_drinks                330
lifetime_spend_hygiene                       330
percentage_of_products_bought_promotion      330
teens_home                                   330
kids_home                                    330
distinct_stores_visited                      330
customer_birthdate                           165
customer_id                                    0
lifetime_spend_groceries                       0
customer_name                                  0
customer_gender                                0
lifetime_spend_nonal

In [17]:
customer_info.duplicated().sum()

np.int64(0)

## DATA QUALITY CHECK

In [18]:
customer_info[
    customer_info['year_first_transaction'] > 2025
][['customer_id', 'year_first_transaction']].head()

,customer_id,year_first_transaction
17,25,2029.0
28,37,2028.0
61,77,2026.0
66,82,2029.0
92,115,2027.0


In [19]:
(customer_info['year_first_transaction'] > 2025).sum()

np.int64(1296)

In [20]:
customer_info[
    (customer_info['percentage_of_products_bought_promotion'] < 0) |
    (customer_info['percentage_of_products_bought_promotion'] > 1)
][[
    'customer_id',
    'percentage_of_products_bought_promotion'
]].head()

,customer_id,percentage_of_products_bought_promotion
8,12,-0.131176
26,35,-0.025524
28,37,-0.184820
65,81,-0.217805
71,87,-0.022180


In [21]:
(
    (
        customer_info['percentage_of_products_bought_promotion'] < 0
    ) |
    (
        customer_info['percentage_of_products_bought_promotion'] > 1
    )
).sum()

np.int64(1755)

In [22]:
customer_info['customer_gender'].value_counts(dropna=False)

customer_gender
female    16577
male      16461
Name: count, dtype: int64

In [23]:
customer_info['customer_birthdate'] = pd.to_datetime(
    customer_info['customer_birthdate'],
    errors='coerce'
)

customer_info['customer_birthdate'].describe()

C:\Users\A K M NAZMUL HOQUE\AppData\Local\Temp\ipykernel_212\2284295263.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  customer_info['customer_birthdate'] = pd.to_datetime(


count                            32873
mean     1971-05-20 20:35:28.913089768
min                1940-01-02 06:03:00
25%                1955-09-04 09:16:00
50%                1971-07-05 14:59:00
75%                1987-03-30 14:07:00
max                2002-01-01 07:25:00
Name: customer_birthdate, dtype: object

In [25]:
customer_info['typical_hour'].describe()

count    32377.000000
mean        12.659388
std          4.854708
min          6.000000
25%          8.000000
50%         12.000000
75%         16.000000
max         23.000000
Name: typical_hour, dtype: float64

In [26]:
customer_info['number_complaints'].value_counts().sort_index()

number_complaints
0.0    10978
1.0    14760
2.0     5271
3.0      914
4.0      235
5.0      167
6.0       45
7.0        7
Name: count, dtype: int64

In [27]:
customer_info['distinct_stores_visited'].describe()

count    32708.000000
mean         3.167941
std          1.674114
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         10.000000
Name: distinct_stores_visited, dtype: float64

In [28]:
spend_cols = [
    'lifetime_spend_groceries',
    'lifetime_spend_electronics',
    'lifetime_spend_vegetables',
    'lifetime_spend_nonalcohol_drinks',
    'lifetime_spend_alcohol_drinks',
    'lifetime_spend_meat',
    'lifetime_spend_fish',
    'lifetime_spend_hygiene',
    'lifetime_spend_videogames',
    'lifetime_spend_petfood'
]

In [29]:
(customer_info[spend_cols] < 0).sum()

lifetime_spend_groceries            0
lifetime_spend_electronics          0
lifetime_spend_vegetables           0
lifetime_spend_nonalcohol_drinks    0
lifetime_spend_alcohol_drinks       0
lifetime_spend_meat                 0
lifetime_spend_fish                 0
lifetime_spend_hygiene              0
lifetime_spend_videogames           0
lifetime_spend_petfood              0
dtype: int64

## DATA CLEANING

In [31]:
customer_clean = customer_info.copy()

In [32]:
customer_clean['percentage_of_products_bought_promotion'] = (
    customer_clean['percentage_of_products_bought_promotion']
    .clip(0, 1)
)

In [33]:
customer_clean['has_loyalty_card'] = (
    customer_clean['loyalty_card_number']
    .notna()
    .astype(int)
)

In [34]:
customer_clean.drop(
    columns=['loyalty_card_number'],
    inplace=True
)

In [35]:
numeric_cols = customer_clean.select_dtypes(
    include=['int64', 'float64']
).columns

customer_clean[numeric_cols] = customer_clean[numeric_cols].fillna(
    customer_clean[numeric_cols].median()
)

In [36]:
customer_clean['customer_birthdate'] = pd.to_datetime(
    customer_clean['customer_birthdate'],
    errors='coerce'
)

In [37]:
median_birthdate = customer_clean['customer_birthdate'].median()

customer_clean['customer_birthdate'] = (
    customer_clean['customer_birthdate']
    .fillna(median_birthdate)
)

In [39]:
customer_clean.isnull().sum().sort_values(ascending=False)

customer_id                                0
customer_name                              0
customer_gender                            0
customer_birthdate                         0
kids_home                                  0
teens_home                                 0
number_complaints                          0
distinct_stores_visited                    0
lifetime_spend_groceries                   0
lifetime_spend_electronics                 0
typical_hour                               0
lifetime_spend_vegetables                  0
lifetime_spend_nonalcohol_drinks           0
lifetime_spend_alcohol_drinks              0
lifetime_spend_meat                        0
lifetime_spend_fish                        0
lifetime_spend_hygiene                     0
lifetime_spend_videogames                  0
lifetime_spend_petfood                     0
lifetime_total_distinct_products           0
percentage_of_products_bought_promotion    0
year_first_transaction                     0
latitude  